# Dynamic Window Models — Classical ML

Train window-level classifiers and regressors on dynamic spectral features.

**Design notes:**
- Features and labels are merged by `song_id` and `window_index`.
- Train/val/test splits come from the existing track-level `split` column (no window-level random splitting).
- Metrics and confusion matrices are saved to `results/metrics/` and `reports/figures/`.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "configs").exists() and (PROJECT_ROOT.parent / "configs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import get_project_root, load_configs, resolve_path
from src.training.train_dynamic_window import (
    load_dynamic_training_data,
    prepare_dynamic_training_frame,
    validate_dynamic_training_frame,
    train_and_evaluate_dynamic_window_models,
)
from src.features.dynamic_window_features import load_dynamic_window_features
from src.data.make_dynamic_dataset import load_dynamic_windows

configs = load_configs(PROJECT_ROOT)
root = get_project_root(PROJECT_ROOT)

## 1. Load dynamic window features and labels

In [2]:
features_df = load_dynamic_window_features(configs)
windows_df = load_dynamic_windows(configs, attach_splits=True, save=False)

print(f"Features: {features_df.shape}")
print(f"Windows: {windows_df.shape}")

Features: (129995, 187)
Windows: (129995, 12)


## 2. Merge and validate modeling table

In [3]:
data = prepare_dynamic_training_frame(features_df, windows_df)
validate_dynamic_training_frame(data)

print(f"Modeling table: {data.shape}")
print(f"Tracks: {data['song_id'].nunique()}")
print("Split counts:")
print(data['split'].value_counts())
data[['song_id', 'window_index', 'split', 'valence', 'arousal', 'dynamic_emotion_quadrant']].head()

Modeling table: (129995, 191)
Tracks: 1802
Split counts:
split
train    94725
val      17952
test     17318
Name: count, dtype: int64


,song_id,window_index,split,valence,arousal,dynamic_emotion_quadrant
0,2,29,train,-0.073341,-0.109386,Q4
1,2,30,train,-0.074001,-0.112164,Q4
2,2,31,train,-0.074369,-0.115677,Q4
3,2,32,train,-0.076115,-0.117513,Q4
4,2,33,train,-0.079871,-0.122535,Q4


## 3. Train classification and regression models

In [4]:
results = train_and_evaluate_dynamic_window_models(configs, data=data)
classification_summary = results["classification"]
regression_summary = results["regression"]

c:\Users\athen\miniconda3\envs\music-emotion-recognition\lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


## 4. Classification results

In [5]:
metrics_dir = resolve_path(root, configs["paths"]["results"]["metrics"])
classification_summary.sort_values(["model_name", "eval_split"])

,model_name,task,target,eval_split,train_size,val_size,test_size,accuracy,macro_f1,weighted_f1,mae,rmse,r2
7,gradient_boosting,classification,dynamic_emotion_quadrant,test,94725,17952,17318,0.599723,0.455348,0.569458,None,None,None
6,gradient_boosting,classification,dynamic_emotion_quadrant,val,94725,17952,17318,0.583723,0.458174,0.555753,None,None,None
1,logistic_regression,classification,dynamic_emotion_quadrant,test,94725,17952,17318,0.600127,0.461823,0.572606,None,None,None
0,logistic_regression,classification,dynamic_emotion_quadrant,val,94725,17952,17318,0.564450,0.434770,0.535214,None,None,None
5,random_forest,classification,dynamic_emotion_quadrant,test,94725,17952,17318,0.603476,0.427674,0.555800,None,None,None
4,random_forest,classification,dynamic_emotion_quadrant,val,94725,17952,17318,0.584670,0.439460,0.544422,None,None,None
3,svm,classification,dynamic_emotion_quadrant,test,94725,17952,17318,0.582342,0.456775,0.563136,None,None,None
2,svm,classification,dynamic_emotion_quadrant,val,94725,17952,17318,0.564004,0.442241,0.541627,None,None,None
9,xgboost,classification,dynamic_emotion_quadrant,test,94725,17952,17318,0.593891,0.451603,0.565774,None,None,None
8,xgboost,classification,dynamic_emotion_quadrant,val,94725,17952,17318,0.588792,0.468683,0.563120,None,None,None


In [6]:
test_clf = classification_summary[classification_summary["eval_split"] == "test"]
test_clf[["model_name", "accuracy", "macro_f1", "weighted_f1"]]

,model_name,accuracy,macro_f1,weighted_f1
1,logistic_regression,0.600127,0.461823,0.572606
3,svm,0.582342,0.456775,0.563136
5,random_forest,0.603476,0.427674,0.555800
7,gradient_boosting,0.599723,0.455348,0.569458
9,xgboost,0.593891,0.451603,0.565774


## 5. Regression results

In [7]:
regression_summary.sort_values(["model_name", "target", "eval_split"])

,model_name,task,target,eval_split,train_size,val_size,test_size,accuracy,macro_f1,weighted_f1,mae,rmse,r2
11,gradient_boosting_regressor,regression,arousal,test,94725,17952,17318,None,None,None,0.155889,0.195622,0.555094
10,gradient_boosting_regressor,regression,arousal,val,94725,17952,17318,None,None,None,0.159583,0.199480,0.462345
9,gradient_boosting_regressor,regression,valence,test,94725,17952,17318,None,None,None,0.173128,0.212223,0.262947
8,gradient_boosting_regressor,regression,valence,val,94725,17952,17318,None,None,None,0.157192,0.192637,0.252171
7,random_forest_regressor,regression,arousal,test,94725,17952,17318,None,None,None,0.158285,0.197677,0.545699
6,random_forest_regressor,regression,arousal,val,94725,17952,17318,None,None,None,0.159512,0.199780,0.460724
5,random_forest_regressor,regression,valence,test,94725,17952,17318,None,None,None,0.172147,0.211847,0.265558
4,random_forest_regressor,regression,valence,val,94725,17952,17318,None,None,None,0.155665,0.190738,0.266838
3,ridge,regression,arousal,test,94725,17952,17318,None,None,None,0.163324,0.202021,0.525512
2,ridge,regression,arousal,val,94725,17952,17318,None,None,None,0.168423,0.212299,0.391025


In [8]:
test_reg = regression_summary[regression_summary["eval_split"] == "test"]
test_reg[["model_name", "target", "mae", "rmse", "r2"]]

,model_name,target,mae,rmse,r2
1,ridge,valence,0.176608,0.215336,0.241169
3,ridge,arousal,0.163324,0.202021,0.525512
5,random_forest_regressor,valence,0.172147,0.211847,0.265558
7,random_forest_regressor,arousal,0.158285,0.197677,0.545699
9,gradient_boosting_regressor,valence,0.173128,0.212223,0.262947
11,gradient_boosting_regressor,arousal,0.155889,0.195622,0.555094


## 6. Saved outputs

In [9]:
figures_dir = resolve_path(root, configs["paths"]["reports"]["figures"])
print("Metrics:", metrics_dir)
print("Figures:", figures_dir)
print("\nClassification summary:", metrics_dir / "dynamic_window_classification_summary.csv")
print("Regression summary:", metrics_dir / "dynamic_window_regression_summary.csv")

Metrics: C:\Users\athen\Desktop\music-emotion-recognition\results\metrics
Figures: C:\Users\athen\Desktop\music-emotion-recognition\reports\figures

Classification summary: C:\Users\athen\Desktop\music-emotion-recognition\results\metrics\dynamic_window_classification_summary.csv
Regression summary: C:\Users\athen\Desktop\music-emotion-recognition\results\metrics\dynamic_window_regression_summary.csv
